# 05 · Chunking
Documents are split into chunks before embedding, because you retrieve *pieces*, not whole files. How you split decides retrieval quality more than almost anything else.

## 1. Fixed-size chunking

In [ ]:
def chunk_fixed(text, size, overlap=0):
    words=text.split(); chunks=[]; step=max(1,size-overlap); i=0
    while i < len(words):
        chunks.append(" ".join(words[i:i+size])); i+=step
    return chunks

para=("Retrieval augmented generation combines a retriever and a generator. The retriever finds "
      "relevant documents for a query. The generator then reads those documents and writes a "
      "grounded answer. Chunking decides what the retriever can find.")
for c in chunk_fixed(para, 12): print("[",c,"]")

## 2. Fixed-size WITH overlap — why it matters
```
  no overlap:  [.....chunk A.....][.....chunk B.....]
                              ^ an idea split HERE is lost from both
  overlap:     [.....chunk A.....]
                          [.....chunk B.....]   <- B repeats A's tail, so the
                                                   boundary idea survives
```

In [ ]:
print("NO overlap:")
for c in chunk_fixed(para, 12, 0): print("  |",c)
print("\nWITH overlap=4 (note repeated words at the seams):")
for c in chunk_fixed(para, 12, 4): print("  |",c)

## 3. Recursive (variable-size) splitting
Split on the biggest natural boundary that fits (paragraph → sentence → word), so chunks stay coherent instead of cutting mid-sentence.

In [ ]:
import re
def chunk_recursive(text, max_len=120):
    # try paragraphs, then sentences, then hard-cut — keep pieces under max_len chars
    def split_units(t, seps):
        if not seps: return [t]
        parts=re.split(seps[0], t)
        out=[]
        for p in parts:
            if len(p)<=max_len: out.append(p)
            else: out.extend(split_units(p, seps[1:]))
        return [p.strip() for p in out if p.strip()]
    return split_units(text, [r"\n\n", r"(?<=[.!?]) ", r" "])

for c in chunk_recursive(para, 90): print("[",c,"]")

**Observe:** fixed chunking is simple but can cut mid-sentence; overlap rescues boundary ideas; recursive splitting respects sentence boundaries so chunks read cleanly. Trade-offs:
```
  smaller chunks -> precise matches, but may lose context
  larger chunks  -> more context, but noisier retrieval
  overlap        -> saves boundary ideas, at a little duplication
```
**Your turn:** re-run notebook 04's metrics after chunking the corpus differently and see which chunking gives better P@K / R@K.